In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time 
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Pipeline de Modelos para Predicción de Accidentes en Barcelona
# Objetivo: Maximizar recall manteniendo precisión decente con datos desbalanceados


# =================== CARGA Y PREPARACIÓN DE DATOS ===================
print("🚀 Iniciando pipeline de predicción de accidentes...")

# Cargar datos
df = pd.read_csv("../data/processed/df_ready_model.csv")
print(f"Dataset cargado: {df.shape[0]:,} registros, {df.shape[1]} columnas")

# Eliminar columna snowfall como solicitado
df = df.drop('snowfall (cm)', axis=1)

# Preparar features para el modelo
# Eliminar columnas no necesarias para el modelo
feature_cols = ['cluster_id', 'temperature_2m (°C)', 'precipitation (mm)', 
                'wind_speed_10m (km/h)', 'Fiesta', 'dia_festivo', 
                'year', 'month', 'day', 'hour', 'day_of_week', 'is_weekend']

X = df[feature_cols].copy()
y = df['accident'].copy()

print(f"Distribución de clases:")
print(f"Clase 0 (No accidente): {(y==0).sum():,} ({(y==0).mean()*100:.2f}%)")
print(f"Clase 1 (Accidente): {(y==1).sum():,} ({(y==1).mean()*100:.2f}%)")

# =================== DIVISIÓN TEMPORAL DE DATOS ===================
# Para series temporales, es importante mantener el orden temporal
# Dividimos por fecha: entrenamiento (2017-2022) y test (2023-2024)

df['year_month'] = df['year'] * 100 + df['month']
train_mask = df['year'] <= 2022
test_mask = df['year'] >= 2023

X_train = X[train_mask].copy()
X_test = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

print(f"\n📊 División temporal:")
print(f"Entrenamiento: {len(X_train):,} registros ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test: {len(X_test):,} registros ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Accidentes en entrenamiento: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Accidentes en test: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

# =================== TÉCNICAS DE BALANCEO OPTIMIZADAS ===================
def create_balanced_datasets(X_train, y_train):
    """Crea datasets balanceados optimizados - SOLO las mejores técnicas"""
    
    balanced_datasets = {}
    
    print("\n⚖️ Creando datasets balanceados (técnicas optimizadas)...")
    
    # 1. Random Under Sampling - Ratio conservador (1:20) - MUY EFICIENTE
    try:
        print("   🔄 Aplicando Under Sampling Conservador (1:20)...")
        rus_conservative = RandomUnderSampler(random_state=42, sampling_strategy=0.05)
        X_rus_cons, y_rus_cons = rus_conservative.fit_resample(X_train, y_train)
        balanced_datasets['undersampling_conservative'] = (X_rus_cons, y_rus_cons)
        print(f"   ✅ Under Sampling Conservador: {len(X_rus_cons):,} registros, {y_rus_cons.sum():,} accidentes ({y_rus_cons.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error en Under Sampling Conservador: {e}")
    
    # 2. Random Under Sampling - Ratio moderado (1:10) - BALANCE ÓPTIMO
    try:
        print("   🔄 Aplicando Under Sampling Moderado (1:10)...")
        rus_moderate = RandomUnderSampler(random_state=42, sampling_strategy=0.1)
        X_rus_mod, y_rus_mod = rus_moderate.fit_resample(X_train, y_train)
        balanced_datasets['undersampling_moderate'] = (X_rus_mod, y_rus_mod)
        print(f"   ✅ Under Sampling Moderado: {len(X_rus_mod):,} registros, {y_rus_mod.sum():,} accidentes ({y_rus_mod.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error en Under Sampling Moderado: {e}")
    
    # 3. Tomek Links - Limpieza de frontera (más conservador)
    try:
        print("   🔄 Aplicando Tomek Links...")
        from imblearn.under_sampling import TomekLinks
        tomek = TomekLinks()
        X_tomek, y_tomek = tomek.fit_resample(X_train, y_train)
        balanced_datasets['tomek_links'] = (X_tomek, y_tomek)
        print(f"   ✅ Tomek Links: {len(X_tomek):,} registros, {y_tomek.sum():,} accidentes ({y_tomek.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error en Tomek Links: {e}")
        
    # 4. Ensemble Undersampling (nuevo) - Combinar múltiples ratios
    try:
        print("   🔄 Aplicando Ensemble Undersampling...")
        # Crear 3 subsets con diferentes ratios y combinarlos
        rus1 = RandomUnderSampler(random_state=42, sampling_strategy=0.08)
        X_ens1, y_ens1 = rus1.fit_resample(X_train, y_train)
        
        rus2 = RandomUnderSampler(random_state=123, sampling_strategy=0.08)
        X_ens2, y_ens2 = rus2.fit_resample(X_train, y_train)
        
        # Combinar eliminando duplicados
        X_ensemble = pd.concat([X_ens1, X_ens2]).drop_duplicates()
        y_ensemble = pd.concat([y_ens1, y_ens2]).loc[X_ensemble.index]
        
        balanced_datasets['ensemble_undersampling'] = (X_ensemble, y_ensemble)
        print(f"   ✅ Ensemble Undersampling: {len(X_ensemble):,} registros, {y_ensemble.sum():,} accidentes ({y_ensemble.mean()*100:.2f}%)")
    except Exception as e:
        print(f"   ❌ Error en Ensemble Undersampling: {e}")
    
    print(f"\n✅ Total de técnicas creadas: {len(balanced_datasets)}")
    print(f"📉 Reducción de combinaciones: De 60 a {len(balanced_datasets) * 6} (usando solo 6 mejores modelos)")
    
    return balanced_datasets

# =================== MODELOS OPTIMIZADOS - SOLO LOS MEJORES ===================
def get_models():
    """Define SOLO los mejores modelos disponibles para máxima eficiencia"""
    
    # Calcular el ratio de desbalance aproximado (será usado para pesos)
    pos_weight = 49  # Aproximado del ratio 98%:2%
    
    models = {}
    models_added = []
    
    print("🤖 Configurando modelos optimizados...")
    
    # 1. Random Forest balanceado - SIEMPRE disponible
    models['RandomForest_Balanced'] = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    models_added.append("RandomForest_Balanced")
    
    # 2. Logistic Regression - SIEMPRE disponible
    models['LogisticRegression_Balanced'] = LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        C=0.1,
        solver='liblinear',
        random_state=42,
        n_jobs=1
    )
    models_added.append("LogisticRegression_Balanced")
    
    # 3. Gradient Boosting - SIEMPRE disponible
    models['GradientBoosting_Tuned'] = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    )
    models_added.append("GradientBoosting_Tuned")
    
    # 4. Random Forest con pesos custom
    models['RandomForest_HighRecall'] = RandomForestClassifier(
        n_estimators=120,
        max_depth=15,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight={0: 1, 1: pos_weight * 1.5},
        random_state=42,
        n_jobs=-1
    )
    models_added.append("RandomForest_HighRecall")
    
    # 5. Intentar XGBoost
    try:
        from xgboost import XGBClassifier
        models['XGBoost_Weighted'] = XGBClassifier(
            n_estimators=150,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=pos_weight,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1,
            verbosity=0
        )
        models_added.append("XGBoost_Weighted")
        print("   ✅ XGBoost añadido")
    except ImportError:
        print("   ⚠️ XGBoost no disponible - usando solo sklearn")
    
    # 6. Intentar LightGBM
    try:
        from lightgbm import LGBMClassifier
        models['LightGBM_Balanced'] = LGBMClassifier(
            n_estimators=150,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            verbose=-1,
            n_jobs=-1
        )
        models_added.append("LightGBM_Balanced")
        print("   ✅ LightGBM añadido")
    except ImportError:
        print("   ⚠️ LightGBM no disponible")
    
    # 7. Intentar CatBoost
    try:
        from catboost import CatBoostClassifier
        models['CatBoost_Weighted'] = CatBoostClassifier(
            iterations=150,
            depth=8,
            learning_rate=0.05,
            l2_leaf_reg=3.0,
            class_weights=[1, pos_weight],
            random_seed=42,
            verbose=False,
            thread_count=-1
        )
        models_added.append("CatBoost_Weighted")
        print("   ✅ CatBoost añadido")
    except ImportError:
        print("   ⚠️ CatBoost no disponible")
    
    print(f"\n📊 Resumen de modelos:")
    print(f"   🎯 Total de modelos: {len(models)}")
    print(f"   📋 Modelos incluidos:")
    for i, model_name in enumerate(models_added, 1):
        print(f"      {i}. {model_name}")
    
    total_combinations = len(models) * 4  # 4 técnicas de balanceo
    print(f"\n📈 Total de combinaciones a evaluar: {total_combinations}")
    print(f"⏱️ Tiempo estimado: {total_combinations * 0.5:.1f}-{total_combinations * 1.5:.1f} minutos")
    
    return models

# =================== EVALUACIÓN AVANZADA ===================
def evaluate_model_comprehensive(model, X_train, X_test, y_train, y_test, model_name, balance_method):
    """Evaluación completa del modelo con métricas específicas para datos desbalanceados"""
    
    # Entrenar modelo
    model.fit(X_train, y_train)
    
    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Métricas
    from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
    
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    results = {
        'model': model_name,
        'balance_method': balance_method,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'roc_auc': roc_auc,
        'true_positives': tp,
        'false_positives': fp,
        'true_negatives': tn,
        'false_negatives': fn,
        'total_accidents_detected': tp,
        'total_accidents_missed': fn,
        'false_alarms': fp
    }
    
    return results, model

# =================== PIPELINE PRINCIPAL ===================
def main_pipeline():
    """Pipeline principal de entrenamiento y evaluación"""
    
    # Crear datasets balanceados
    balanced_datasets = create_balanced_datasets(X_train, y_train)
    
    # Obtener modelos
    models = get_models()
    
    # Normalizar datos (importante para algunos modelos)
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Lista para almacenar resultados
    all_results = []
    trained_models = {}
    
    print(f"\n🔬 Evaluando combinaciones optimizadas...")
    print(f"    📊 Combinaciones: {len(models)} modelos × {len(balanced_datasets)} técnicas = {len(models) * len(balanced_datasets)}")
    print(f"    🎯 Objetivo: Maximizar RECALL con eficiencia computacional")
    print(f"    ⚡ Optimizaciones: Paralelización + datasets reducidos + modelos eficientes")
    print(f"    🚫 Excluidos: Dataset original (2%) + Under Sampling Agresivo (1:5)")
    
    # Evaluar cada combinación
    total_combinations = len(models) * len(balanced_datasets)
    current = 0
    start_time = time.time()
    
    for balance_name, (X_bal, y_bal) in balanced_datasets.items():
        
        print(f"\n  📊 Técnica: {balance_name.upper()}")
        print(f"      Datos: {len(X_bal):,} registros ({y_bal.mean()*100:.2f}% accidentes)")
        
        # Escalar datos balanceados si es necesario
        X_bal_scaled = scaler.transform(X_bal)
        
        for model_name, model in models.items():
            current += 1
            elapsed = time.time() - start_time
            eta = (elapsed / current) * (total_combinations - current) if current > 0 else 0
            
            print(f"    [{current:2d}/{total_combinations}] {model_name:20s} (ETA: {eta/60:.1f}m)", end=" → ")
            
            try:
                # Usar datos escalados para modelos sensibles a la escala
                if 'LogisticRegression' in model_name:
                    X_train_use = X_bal_scaled
                    X_test_use = X_test_scaled
                else:
                    X_train_use = X_bal
                    X_test_use = X_test
                
                # Entrenar con timeout implícito (modelos optimizados)
                results, trained_model = evaluate_model_comprehensive(
                    model, X_train_use, X_test_use, y_bal, y_test, 
                    model_name, balance_name
                )
                
                all_results.append(results)
                trained_models[f"{model_name}_{balance_name}"] = trained_model
                
                # Mostrar resultados con código de colores
                recall_color = "🟢" if results['recall'] > 0.7 else "🟡" if results['recall'] > 0.5 else "🔴"
                precision_color = "🟢" if results['precision'] > 0.2 else "🟡" if results['precision'] > 0.1 else "🔴"
                
                print(f"{recall_color}R:{results['recall']:.3f} {precision_color}P:{results['precision']:.3f} F1:{results['f1_score']:.3f}")
                
            except Exception as e:
                print(f"❌ Error: {str(e)[:30]}...")
                continue
    
    total_time = time.time() - start_time
    print(f"\n⏱️  Tiempo total de entrenamiento: {total_time/60:.1f} minutos")
    
    return all_results, trained_models, scaler

# =================== ANÁLISIS DE RESULTADOS ===================
def analyze_results(results):
    """Analiza y presenta los resultados de manera comprensible - FOCO EN NO-SINTÉTICOS"""
    
    df_results = pd.DataFrame(results)
    
    # Ordenar por recall (objetivo principal) y luego por precision
    df_results = df_results.sort_values(['recall', 'precision'], ascending=[False, False])
    
    print(f"\n📈 RESULTADOS COMPLETOS - {len(df_results)} COMBINACIONES EVALUADAS")
    print("=" * 120)
    print("🎯 OBJETIVO: Maximizar detección de accidentes (RECALL) usando SOLO datos reales")
    print("📊 MÉTRICAS: Recall=Sensibilidad | Precision=Valor Predictivo Positivo | F1=Balance")
    print("=" * 120)
    
    # Estadísticas generales
    print(f"\n📊 ESTADÍSTICAS GENERALES:")
    print(f"   🎯 Recall promedio: {df_results['recall'].mean():.3f} (rango: {df_results['recall'].min():.3f} - {df_results['recall'].max():.3f})")
    print(f"   📊 Precision promedio: {df_results['precision'].mean():.3f} (rango: {df_results['precision'].min():.3f} - {df_results['precision'].max():.3f})")
    print(f"   ⚖️  F1-Score promedio: {df_results['f1_score'].mean():.3f} (rango: {df_results['f1_score'].min():.3f} - {df_results['f1_score'].max():.3f})")
    
    # Top 15 modelos
    print(f"\n🏆 TOP 15 MEJORES MODELOS (ordenados por Recall):")
    print("-" * 120)
    
    top_models = df_results.head(15)
    
    print(f"{'#':>2} {'MODELO':^25} {'BALANCEO':^20} {'RECALL':^8} {'PREC':^8} {'F1':^8} {'DETECTA':^8} {'PIERDE':^8} {'F.ALARM':^8}")
    print("-" * 120)
    
    for i, (idx, row) in enumerate(top_models.iterrows(), 1):
        modelo_corto = row['model'][:24]
        balance_corto = row['balance_method'][:19]
        
        print(f"{i:2d} {modelo_corto:25} {balance_corto:20} "
              f"{row['recall']:8.3f} {row['precision']:8.3f} {row['f1_score']:8.3f} "
              f"{row['total_accidents_detected']:8,} {row['total_accidents_missed']:8,} {row['false_alarms']:8,}")
    
    # Análisis por técnica de balanceo
    print(f"\n📊 RENDIMIENTO POR TÉCNICA DE BALANCEO:")
    print("-" * 80)
    
    balance_analysis = df_results.groupby('balance_method').agg({
        'recall': ['mean', 'max', 'min'],
        'precision': ['mean', 'max', 'min'],
        'f1_score': ['mean', 'max', 'min']
    }).round(3)
    
    for balance_method in df_results['balance_method'].unique():
        subset = df_results[df_results['balance_method'] == balance_method]
        print(f"  🔹 {balance_method:25} | Modelos: {len(subset):2d} | "
              f"Recall: {subset['recall'].mean():.3f}±{subset['recall'].std():.3f} | "
              f"Precision: {subset['precision'].mean():.3f}±{subset['precision'].std():.3f}")
    
    # Análisis por tipo de modelo
    print(f"\n🤖 RENDIMIENTO POR TIPO DE ALGORITMO:")
    print("-" * 80)
    
    # Extraer familia de algoritmo (antes del primer _)
    df_results['model_family'] = df_results['model'].str.split('_').str[0]
    
    for model_family in df_results['model_family'].unique():
        subset = df_results[df_results['model_family'] == model_family]
        best_combo = subset.iloc[0]
        print(f"  🔸 {model_family:15} | Combos: {len(subset):2d} | "
              f"Mejor Recall: {subset['recall'].max():.3f} | "
              f"Mejor combo: {best_combo['balance_method']:15} | "
              f"F1: {best_combo['f1_score']:.3f}")
    
    # Recomendaciones específicas
    print(f"\n💡 RECOMENDACIONES BASADAS EN RESULTADOS:")
    print("-" * 80)
    
    best_overall = df_results.iloc[0]
    best_precision = df_results.loc[df_results['precision'].idxmax()]
    best_f1 = df_results.loc[df_results['f1_score'].idxmax()]
    
    print(f"  🥇 MEJOR RECALL (detectar más accidentes):")
    print(f"      {best_overall['model']} + {best_overall['balance_method']}")
    print(f"      Detecta {best_overall['total_accidents_detected']:,} accidentes de {best_overall['total_accidents_detected'] + best_overall['total_accidents_missed']:,} "
          f"({(best_overall['total_accidents_detected']/(best_overall['total_accidents_detected'] + best_overall['total_accidents_missed']))*100:.1f}%)")
    print(f"      Genera {best_overall['false_alarms']:,} falsas alarmas")
    
    print(f"\n  🎯 MEJOR PRECISIÓN (menos falsas alarmas):")
    print(f"      {best_precision['model']} + {best_precision['balance_method']}")
    print(f"      Precisión: {best_precision['precision']:.3f} | Recall: {best_precision['recall']:.3f}")
    
    print(f"\n  ⚖️  MEJOR BALANCE F1 (equilibrio general):")
    print(f"      {best_f1['model']} + {best_f1['balance_method']}")
    print(f"      F1: {best_f1['f1_score']:.3f} | Recall: {best_f1['recall']:.3f} | Precision: {best_f1['precision']:.3f}")
    
    # Insights específicos sobre técnicas sin sintéticos
    print(f"\n🔍 INSIGHTS SOBRE TÉCNICAS SIN DATOS SINTÉTICOS:")
    print("-" * 80)
    
    # Mejor técnica de undersampling
    undersampling_methods = [col for col in df_results['balance_method'].unique() if 'undersampling' in col]
    if undersampling_methods:
        best_under = df_results[df_results['balance_method'].isin(undersampling_methods)].iloc[0]
        print(f"  📉 Mejor Undersampling: {best_under['balance_method']}")
        print(f"      Recall: {best_under['recall']:.3f} | Reduce dataset a ~{best_under['total_accidents_detected'] + best_under['total_accidents_missed'] + best_under['false_positives'] + best_under['true_negatives']:,} registros")
    
    # Efectividad de pesos vs undersampling
    original_best = df_results[df_results['balance_method'] == 'original'].iloc[0] if len(df_results[df_results['balance_method'] == 'original']) > 0 else None
    if original_best is not None:
        print(f"\n  ⚖️  Comparación Original vs Mejor Técnica:")
        improvement_recall = ((best_overall['recall'] - original_best['recall']) / original_best['recall']) * 100
        print(f"      Mejora en Recall: {improvement_recall:+.1f}%")
        print(f"      Original: {original_best['recall']:.3f} → Mejor: {best_overall['recall']:.3f}")
    
    return df_results

# =================== OPTIMIZACIÓN DEL MEJOR MODELO ===================
def optimize_best_model(best_model_name, best_balance_method, balanced_datasets, models):
    """Optimiza hiperparámetros del mejor modelo"""
    
    print(f"\n🎯 Optimizando {best_model_name} con {best_balance_method}...")
    
    X_opt, y_opt = balanced_datasets[best_balance_method]
    base_model = models[best_model_name]
    
    # Definir grids de parámetros según el modelo
    param_grids = {
        'RandomForest': {
            'n_estimators': [100, 200],
            'max_depth': [8, 12, 16],
            'min_samples_split': [5, 10, 20],
            'min_samples_leaf': [2, 5, 10]
        },
        'XGBoost': {
            'n_estimators': [100, 200],
            'max_depth': [4, 6, 8],
            'learning_rate': [0.05, 0.1, 0.2],
            'subsample': [0.7, 0.8, 0.9]
        },
        'LightGBM': {
            'n_estimators': [100, 200],
            'max_depth': [4, 6, 8],
            'learning_rate': [0.05, 0.1, 0.2],
            'subsample': [0.7, 0.8, 0.9]
        }
    }
    
    if best_model_name in param_grids:
        param_grid = param_grids[best_model_name]
        
        # Usar TimeSeriesSplit para validación
        tscv = TimeSeriesSplit(n_splits=3)
        
        # GridSearch con scoring enfocado en recall
        grid_search = GridSearchCV(
            base_model,
            param_grid,
            cv=tscv,
            scoring='recall',
            n_jobs=-1,
            verbose=1
        )
        
        grid_search.fit(X_opt, y_opt)
        
        print(f"✅ Mejor combinación de parámetros:")
        for param, value in grid_search.best_params_.items():
            print(f"  {param}: {value}")
        
        return grid_search.best_estimator_
    
    else:
        print(f"  ℹ️  Optimización no disponible para {best_model_name}")
        return base_model

# =================== EJECUCIÓN PRINCIPAL ===================
if __name__ == "__main__":
    # Ejecutar pipeline principal
    results, trained_models, scaler = main_pipeline()
    
    # Analizar resultados
    df_results = analyze_results(results)
    
    # Guardar resultados detallados
    df_results.to_csv("../results/model_comparison_results.csv", index=False)
    print(f"\n💾 Resultados guardados en: ../results/model_comparison_results.csv")
    
    # Obtener el mejor modelo
    best_result = df_results.iloc[0]
    best_model_name = best_result['model']
    best_balance_method = best_result['balance_method']
    
    print(f"\n🏆 MEJOR MODELO ENCONTRADO:")
    print(f"Modelo: {best_model_name}")
    print(f"Método de balanceo: {best_balance_method}")
    print(f"Recall: {best_result['recall']:.3f}")
    print(f"Precision: {best_result['precision']:.3f}")
    print(f"F1-Score: {best_result['f1_score']:.3f}")
    
    # Mostrar recomendaciones finales
    print(f"\n💡 RECOMENDACIONES:")
    print(f"1. Tu mejor modelo detecta {best_result['total_accidents_detected']} de {best_result['total_accidents_detected'] + best_result['false_negatives']} accidentes")
    print(f"2. Genera {best_result['false_alarms']:,} falsas alarmas")
    print(f"3. Para uso en producción, considera ajustar el threshold de decisión si necesitas más recall o menos falsas alarmas")
    
    # Opcional: optimizar el mejor modelo
    print(f"\n🔧 ¿Deseas optimizar el mejor modelo? (Comentar/descomentar las siguientes líneas)")
    
    # balanced_datasets = create_balanced_datasets(X_train, y_train)
    # models = get_models()
    # optimized_model = optimize_best_model(best_model_name, best_balance_method, balanced_datasets, models)
    
    print("\n✅ Pipeline completado exitosamente!")

🚀 Iniciando pipeline de predicción de accidentes...
Dataset cargado: 3,495,160 registros, 16 columnas
Distribución de clases:
Clase 0 (No accidente): 3,427,736 (98.07%)
Clase 1 (Accidente): 67,424 (1.93%)

📊 División temporal:
Entrenamiento: 2,604,597 registros (74.5%)
Test: 890,563 registros (25.5%)
Accidentes en entrenamiento: 52,167 (2.00%)
Accidentes en test: 15,257 (1.71%)

⚖️ Creando datasets balanceados (técnicas optimizadas)...
   🔄 Aplicando Under Sampling Conservador (1:20)...
   ✅ Under Sampling Conservador: 1,095,507 registros, 52,167 accidentes (4.76%)
   🔄 Aplicando Under Sampling Moderado (1:10)...
   ✅ Under Sampling Moderado: 573,837 registros, 52,167 accidentes (9.09%)
   🔄 Aplicando Tomek Links...
   ✅ Tomek Links: 2,604,541 registros, 52,167 accidentes (2.00%)
   🔄 Aplicando Ensemble Undersampling...
   ✅ Ensemble Undersampling: 1,146,070 registros, 59,682 accidentes (4.45%)

✅ Total de técnicas creadas: 4
📉 Reducción de combinaciones: De 60 a 24 (usando solo 6 mejo

: 